In [2]:
import cv2
import numpy as np
import os
import glob


## Calibration

In [3]:
# Configuration
PATTERN = (8, 6)         # Checkerboard INNER corners
SQUARE_M = 0.0225        # Square size in meters (22.5 mm)
MIN_PAIRS = 5            # Minimum valid pairs (15-30 strongly advised for phone sensors)
ALPHA = 0.8              # 1.0: retain all valid pixels (no heavy zoom-in crop); 0.0: aggressive crop

CAPTURE_DIR_LEFT = "captured_pairs/left"
CAPTURE_DIR_RIGHT = "captured_pairs/right"
OUTPUT_NPZ = "stereo_calib.npz"

# 3D coordinates of the inner corners on the checkerboard plane (z = 0)
# OpenCV standard: row-major order (y along rows, x along columns)
objp = np.zeros((PATTERN[0] * PATTERN[1], 3), np.float32)
for y in range(PATTERN[1]):
    for x in range(PATTERN[0]):
        objp[y * PATTERN[0] + x] = [x * SQUARE_M, y * SQUARE_M, 0.0]


# Ensure corner 0 in the left frame matches corner 0 in the right frame
# If the detection along row 0 runs in opposite directions, one view is flipped 180
def align_corner_ordering(cl, cr):
    v_l = cl[1][0] - cl[0][0]
    v_r = cr[1][0] - cr[0][0]

    # If the direction vectors point opposite (dot product < 0), flip right corner sequence
    if np.dot(v_l, v_r) < 0:
        cr = np.ascontiguousarray(np.flip(cr, axis=0))
    return cl, cr


In [4]:
# Load images and verify pairs
left_images = sorted(glob.glob(os.path.join(CAPTURE_DIR_LEFT, "*.jpg")))
right_images = sorted(glob.glob(os.path.join(CAPTURE_DIR_RIGHT, "*.jpg")))

if len(left_images) == 0 or len(left_images) != len(right_images):
    raise SystemExit(
        f"Error: Found {len(left_images)} left images and {len(right_images)} right images"
        "Lists must be non-empty and equal in count"
    )

print(f"Discovered {len(left_images)} image pairs, processing:")

objpoints = []
impoints_l = []
impoints_r = []
pair_names = []
img_size = None

# Sector-based detection flags, robust against high-res phone JPG edge ringing
sb_flags = (
    cv2.CALIB_CB_EXHAUSTIVE
    | cv2.CALIB_CB_ACCURACY
    | cv2.CALIB_CB_NORMALIZE_IMAGE
)

# Corner detection and synchronisation
for left_path, right_path in zip(left_images, right_images):
    pair_id = os.path.basename(left_path)
    l = cv2.imread(left_path)
    r = cv2.imread(right_path)

    if l is None or r is None:
        print(f"  [x] Failed to read image pair: {pair_id}")
        continue

    gl = cv2.cvtColor(l, cv2.COLOR_BGR2GRAY)
    gr = cv2.cvtColor(r, cv2.COLOR_BGR2GRAY)

    if img_size is None:
        img_size = (gl.shape[1], gl.shape[0])  # (width, height)

    okl, cl = cv2.findChessboardCornersSB(gl, PATTERN, flags=sb_flags)
    okr, cr = cv2.findChessboardCornersSB(gr, PATTERN, flags=sb_flags)

    if okl and okr:
        cl = cl.astype(np.float32)
        cr = cr.astype(np.float32)

        # Enforce consistent coordinate start and direction across stereo pairs
        cl, cr = align_corner_ordering(cl, cr)

        objpoints.append(objp)
        impoints_l.append(cl)
        impoints_r.append(cr)
        pair_names.append(pair_id)
        print(f"  [+] Corners synced and extracted: {pair_id}")
    else:
        print(f"  [x] Pattern not found in both views: {pair_id}")

if len(objpoints) < MIN_PAIRS:
    raise SystemExit(
        f"Only {len(objpoints)} valid pairs extracted, at least {MIN_PAIRS} required"
    )


Discovered 19 image pairs, processing:
  [x] Pattern not found in both views: left_017.jpg
  [x] Pattern not found in both views: left_018.jpg
  [+] Corners synced and extracted: left_019.jpg
  [+] Corners synced and extracted: left_020.jpg
  [x] Pattern not found in both views: left_021.jpg
  [+] Corners synced and extracted: left_022.jpg
  [+] Corners synced and extracted: left_023.jpg
  [+] Corners synced and extracted: left_024.jpg
  [+] Corners synced and extracted: left_025.jpg
  [+] Corners synced and extracted: left_026.jpg
  [+] Corners synced and extracted: left_027.jpg
  [+] Corners synced and extracted: left_028.jpg
  [x] Pattern not found in both views: left_029.jpg
  [+] Corners synced and extracted: left_030.jpg
  [+] Corners synced and extracted: left_031.jpg
  [+] Corners synced and extracted: left_032.jpg
  [+] Corners synced and extracted: left_033.jpg
  [+] Corners synced and extracted: left_034.jpg
  [+] Corners synced and extracted: left_035.jpg


In [5]:
# Monocular camera calibration
print(f"\nCalibrating intrinsics on {len(objpoints)} pairs:")
w, h = img_size
focal_init = max(w, h)  # Standard ~1x smartphone lens prior

K1_init = np.array([[focal_init, 0.0, w / 2.0],
                    [0.0, focal_init, h / 2.0],
                    [0.0, 0.0, 1.0]], dtype=np.float64)
K2_init = K1_init.copy()

mono_flags = (
    cv2.CALIB_USE_INTRINSIC_GUESS
    | cv2.CALIB_FIX_ASPECT_RATIO
    | cv2.CALIB_ZERO_TANGENT_DIST
)

rmsL, K1, D1, _, _ = cv2.calibrateCamera(objpoints, impoints_l, img_size, K1_init, None, flags=mono_flags)
rmsR, K2, D2, _, _ = cv2.calibrateCamera(objpoints, impoints_r, img_size, K2_init, None, flags=mono_flags)

print(f"Left Camera RMS:   {rmsL:.3f} px")
print(f"Right Camera RMS:  {rmsR:.3f} px")

# Stereo Extrinsic Calibration
print("\nSolving stereo extrinsics (R, T)")
crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-6)

# Preserve accurate mono intrinsics, solve purely for spatial pose (R, T)
rms_stereo, K1, D1, K2, D2, R, T, E, F = cv2.stereoCalibrate(
    objpoints, impoints_l, impoints_r,
    K1, D1, K2, D2,
    img_size,
    criteria=crit,
    flags=cv2.CALIB_FIX_INTRINSIC
)

baseline_m = np.linalg.norm(T)
print(f"Stereo RMS error:  {rms_stereo:.3f} px")
print(f"Baseline distance: {baseline_m * 1000.0:.2f} mm")
print("\nRotation Matrix R:\n", R)
print("\nTranslation Vector T (m):\n", T.ravel())

# Stereo Rectification Parameters
R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
    K1, D1, K2, D2, img_size, R, T,
    flags=cv2.CALIB_ZERO_DISPARITY,
    alpha=ALPHA,    # alpha=1.0 retains full FOV with black margins, alpha=0 crops heavily & cause zoom in issue
    newImageSize=img_size
)

np.savez(
    OUTPUT_NPZ,
    K1=K1, D1=D1, K2=K2, D2=D2,
    R=R, T=T, E=E, F=F,
    R1=R1, R2=R2, P1=P1, P2=P2, Q=Q,
    roi1=roi1, roi2=roi2,
    img_size=img_size
)

print(f"\nCalibration data and undistort-rectify maps saved to '{OUTPUT_NPZ}'")


Calibrating intrinsics on 15 pairs:
Left Camera RMS:   0.547 px
Right Camera RMS:  0.552 px

Solving stereo extrinsics (R, T)
Stereo RMS error:  2.080 px
Baseline distance: 68.68 mm

Rotation Matrix R:
 [[ 9.99993497e-01 -1.64454122e-04 -3.60251555e-03]
 [ 1.57025479e-04  9.99997861e-01 -2.06225669e-03]
 [ 3.60284699e-03  2.06167759e-03  9.99991384e-01]]

Translation Vector T (m):
 [-0.06866441 -0.00142103 -0.00027635]

Calibration data and undistort-rectify maps saved to 'stereo_calib.npz'


## Reconstruction

In [6]:
# Reconstruction

calib = np.load("stereo_calib.npz")
K1, D1, K2, D2 = calib["K1"], calib["D1"], calib["K2"], calib["D2"]
R1, R2, P1, P2, Q = calib["R1"], calib["R2"], calib["P1"], calib["P2"], calib["Q"]
img_size = tuple(calib["img_size"])  # (width, height)

# Precompute remap lookup tables
map1l, map2l = cv2.initUndistortRectifyMap(K1, D1, R1, P1, img_size, cv2.CV_32FC1)
map1r, map2r = cv2.initUndistortRectifyMap(K2, D2, R2, P2, img_size, cv2.CV_32FC1)

INPUT_DIR = "reconstruction_capture"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load images of scene to be reconstructed
left_file = os.path.join(INPUT_DIR, "raw_left_04.jpg")
right_file = os.path.join(INPUT_DIR, "raw_right_04.jpg")

l = cv2.imread(left_file)
r = cv2.imread(right_file)
if l is None or r is None:
    raise FileNotFoundError("Could not find calibration image pairs")

# Rectify at full resolution
rl = cv2.remap(l, map1l, map2l, cv2.INTER_LINEAR)
rr = cv2.remap(r, map1r, map2r, cv2.INTER_LINEAR)
cv2.imwrite(os.path.join(OUTPUT_DIR, "rectified_pair.png"), np.hstack([rl, rr]))


True

In [7]:
# Downscale for stereo matching (phone resolutions are too wide for standard numDisparities)
# Scale factor: 0.5 (e.g., 4000x3000 -> 2000x1500) or 0.25
SCALE = 0.5

# Configure SGBM with robust phone-resolution parameters
# numDisparities must be divisible by 16
NUM_DISPARITIES = 16 * 16  # 256 disparities covers close-range objects at 0.5x scale
BLOCK_SIZE = 7

sgbm = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=NUM_DISPARITIES,
    blockSize=BLOCK_SIZE,
    P1=8 * 3 * BLOCK_SIZE**2,
    P2=64 * 3 * BLOCK_SIZE**2,  # Stronger spatial regularization to reject JPG noise
    disp12MaxDiff=2,
    uniquenessRatio=15,         # Higher uniqueness (15) filters out low-texture flat surfaces
    speckleWindowSize=150,      # Kill floating speckle clusters
    speckleRange=2,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
)

# Downscale by SCALE
target_w = int(img_size[0] * SCALE)
target_h = int(img_size[1] * SCALE)
rl_small = cv2.resize(rl, (target_w, target_h), interpolation=cv2.INTER_AREA)
rr_small = cv2.resize(rr, (target_w, target_h), interpolation=cv2.INTER_AREA)

# Disparity computation
gray_l = cv2.cvtColor(rl_small, cv2.COLOR_BGR2GRAY)
gray_r = cv2.cvtColor(rr_small, cv2.COLOR_BGR2GRAY)
disp = (sgbm.compute(gray_l, gray_r).astype(np.float32) / 16.0)

# Zero out border margins to prevent edge asymptote explosion
BORDER_CROP = int(target_w * 0.06)  # apply margin to all sides
disp[:BORDER_CROP, :] = 0
disp[-BORDER_CROP:, :] = 0
disp[:, :BORDER_CROP] = 0
disp[:, -BORDER_CROP:] = 0



# Adjust Q matrix for the downscaled disparity map
Q_scaled = Q.copy()
Q_scaled[0, 3] *= SCALE   # -cx
Q_scaled[1, 3] *= SCALE   # -cy
Q_scaled[2, 3] *= SCALE   # f
Q_scaled[3, 2] /= SCALE   # -1 / baseline


In [8]:
# Reproject to 3D
points = cv2.reprojectImageTo3D(disp, Q_scaled)
colors = cv2.cvtColor(rl_small, cv2.COLOR_BGR2RGB)

# Based on setup, adjust to produce coherent point cloud
Z_MIN = 0.10   # min m close
Z_MAX = 0.70   # max m away
X_MAX = 0.60   # +/- max m left/right
Y_MAX = 0.50   # +/- max m up/down

"""
# Causes kernel crash
x = points[:, :, 0]
y = points[:, :, 1]
z = points[:, :, 2]

mask = (
    (disp > 4.0) &
    np.isfinite(x) & np.isfinite(y) & np.isfinite(z) &
    (z >= Z_MIN) & (z <= Z_MAX) &
    (np.abs(x) <= X_MAX) &
    (np.abs(y) <= Y_MAX)
)
"""

# Reshape points and colors to (N, 3) contiguous arrays
# Avoids non-contiguous 3D slice traps to prevent memory caused kernel crash
flat_points = np.ascontiguousarray(points.reshape(-1, 3))
flat_colors = np.ascontiguousarray(colors.reshape(-1, 3))
flat_disp = disp.ravel()

# Extract 1D vectors
x = flat_points[:, 0]
y = flat_points[:, 1]
z = flat_points[:, 2]

# Mask step 1: Disparity and finiteness check ONLY
# Filters out Inf, NaN, and denormals BEFORE any math or comparisons on them
mask = (flat_disp > 4.0) & np.isfinite(z) & np.isfinite(x) & np.isfinite(y)

# Mask Step 2: Apply physical range checks in-place ONLY on the remaining points
# Prevents NaN/Inf from ever entering < or > comparisons
mask &= (z >= Z_MIN) & (z <= Z_MAX)
mask &= (x >= -X_MAX) & (x <= X_MAX)
mask &= (y >= -Y_MAX) & (y <= Y_MAX)

# Extract valid points
pts = flat_points[mask]
cls = flat_colors[mask]

print(f"Retained point count: {len(pts)}")
print(f"X bounds: [{pts[:, 0].min():.3f}, {pts[:, 0].max():.3f}] m")
print(f"Y bounds: [{pts[:, 1].min():.3f}, {pts[:, 1].max():.3f}] m")
print(f"Z bounds: [{pts[:, 2].min():.3f}, {pts[:, 2].max():.3f}] m")

# Coordinate Adjustment for 3D Viewers (Invert Y and Z if scene appears upside down)
pts[:, 1] = -pts[:, 1]
pts[:, 2] = -pts[:, 2]

# Save PLY
header = (
    "ply\nformat ascii 1.0\nelement vertex %d\n"
    "property float x\nproperty float y\nproperty float z\n"
    "property uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n"
)

out_ply = os.path.join(OUTPUT_DIR, "new_scene.ply")
with open(out_ply, "wb") as fo:
    fo.write((header % len(pts)).encode())
    np.savetxt(fo, np.hstack([pts, cls]), fmt="%.4f %.4f %.4f %d %d %d")

print(f"PLY successfully saved: {out_ply} ({len(pts)} points)")


Retained point count: 445726
X bounds: [-0.345, 0.517] m
Y bounds: [-0.190, 0.167] m
Z bounds: [0.164, 0.699] m
PLY successfully saved: outputs/new_scene.ply (445726 points)


In [9]:
# Save Disparity Color Preview
dv = np.clip(disp, 0, NUM_DISPARITIES)
# dv[~valid] = 0
dv_norm = cv2.normalize(dv, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
disp_color = cv2.applyColorMap(dv_norm, cv2.COLORMAP_TURBO)


# Generate colour legend
H, W = disp_color.shape[:2]
padding_height = 15
legend_height = 30
text_height = 30

# Padding
padding_area = np.zeros((padding_height, W, 3), dtype=np.uint8)

# Create a horizontal gradient from 255 down to 0 (Closer/Red on Left, Further/Blue on Right)
gradient = np.linspace(255, 0, W, dtype=np.uint8)
gradient_img = np.tile(gradient, (legend_height, 1))
legend_bar = cv2.applyColorMap(gradient_img, cv2.COLORMAP_TURBO)

# Create bar for text labels
text_area = np.zeros((text_height, W, 3), dtype=np.uint8)
font = cv2.FONT_HERSHEY_SIMPLEX

# Add "Closer" label to the left (Red / High Disparity)
cv2.putText(text_area, "Closer", (10, 22), font, 0.7, (255, 255, 255), 1, cv2.LINE_AA)

# Add "Further" label to the right (Blue / Low Disparity)
text_further = "Further"
text_size = cv2.getTextSize(text_further, font, 0.7, 1)[0]
cv2.putText(text_area, text_further, (W - text_size[0] - 10, 22), font, 0.7, (255, 255, 255), 1, cv2.LINE_AA)

# Stack disparity map, padding, colorbar, and text vertically
disparity_final = np.vstack([disp_color, padding_area, legend_bar, text_area])


cv2.imwrite(os.path.join(OUTPUT_DIR, "disparity_preview.png"), disparity_final)
print("Disparity preview saved")

Disparity preview saved
